In [8]:

from flask import Flask, request, jsonify
import joblib
import pandas as pd
import numpy as np
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
app = Flask(__name__)

model = joblib.load("model/salary_model.pkl")
feature_cols = joblib.load("model/feature_cols.pkl")
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def build_input_vector(data):
    """Turn form inputs into the same feature vector the model was trained on."""
    row = {col: 0 for col in feature_cols}

    exp_map = {"Entry": 0, "Mid": 1, "Senior": 2, "Executive": 3}
    emp_map = {"Part-time": 0, "Freelance": 1, "Contract": 2, "Full-time": 3}
    size_map = {"Small": 0, "Medium": 1, "Large": 2}

    row["experience_level"] = exp_map.get(data.get("experience_level"), 1)
    row["employment_type"] = emp_map.get(data.get("employment_type"), 3)
    row["company_size"] = size_map.get(data.get("company_size"), 1)
    row["remote_ratio"] = int(data.get("remote_ratio", 50))
    row["work_year"] = int(data.get("work_year", 2023))

    job_col = f"job_title_{data.get('job_title', 'Data Scientist')}"
    if job_col in row:
        row[job_col] = 1

    residence_col = f"employee_residence_{data.get('residence', 'US')}"
    if residence_col in row:
        row[residence_col] = 1

    location_col = f"company_location_{data.get('company_location', 'US')}"
    if location_col in row:
        row[location_col] = 1

    return pd.DataFrame([row])

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    X = build_input_vector(data)
    predicted = model.predict(X)[0]
    predicted = max(0, round(predicted, 2))
    return jsonify({"predicted_salary": predicted})

@app.route("/suggest", methods=["POST"])
def suggest():
    data = request.json
    predicted = data.get("predicted_salary", 0)
    job_title = data.get("job_title", "Data Scientist")
    experience = data.get("experience_level", "Mid")
    location = data.get("residence", "US")
    remote = data.get("remote_ratio", 50)

    prompt = f"""You are a senior career coach for data professionals.

A {experience}-level {job_title} based in {location} (remote ratio: {remote}%) 
has a predicted annual salary of ${predicted:,.0f} USD.

Give exactly 3 specific, actionable suggestions to help them increase their 
salary by 20-30% within 12 months. Be concrete — mention real skills, 
certifications, or strategies. Keep each suggestion to 2-3 sentences.
Format as: 1. [suggestion] 2. [suggestion] 3. [suggestion]"""

    message = client.messages.create(
        model="claude-opus-4-20250514",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    return jsonify({"suggestions": message.content[0].text})

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

if __name__ == "__main__":
    app.run(debug=True, port=5000)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (fsevents)
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

SystemExit: 1

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
